In [15]:
import sys
import subprocess

def install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--user", pkg])

packages = ["flask", "requests", "redis", "python-dotenv", "flask-limiter"]
for p in packages:
    try:
        __import__(p.split('-')[0])
    except ImportError:
        install(p)


In [19]:
import os

# Set your API key and Redis URL here
os.environ["WEATHER_API_KEY"] = "your_actual_key_here"

os.environ["REDIS_URL"] = "redis://localhost:6379"


In [20]:
import os
import json
import requests
import redis
from flask import Flask, jsonify
from flask_limiter import Limiter
from flask_limiter.util import get_remote_address

# Load environment variables
WEATHER_API_KEY = os.getenv("WEATHER_API_KEY")
REDIS_URL = os.getenv("REDIS_URL")

# Connect to Redis
r = redis.Redis.from_url(REDIS_URL)

# Flask app
app = Flask(__name__)

# In-memory rate limiter (safe fallback if Redis storage module isn't available)
limiter = Limiter(
    get_remote_address,
    app=app,
    default_limits=["5 per minute"]
)

@app.route("/")
def home():
    return jsonify({"message": "✅ Weather API is running!"})

@app.route("/weather/<city>", methods=["GET"])
@limiter.limit("10 per hour")
def get_weather(city):
    if not WEATHER_API_KEY:
        return jsonify({"error": "Missing API key"}), 500

    cache_key = f"weather:{city.lower()}"
    cached_data = r.get(cache_key)

    if cached_data:
        print("✅ Data served from cache")
        return jsonify({"source": "cache", "data": json.loads(cached_data)})

    try:
        url = (
            f"https://weather.visualcrossing.com/VisualCrossingWebServices/rest/services/"
            f"timeline/{city}?unitGroup=metric&key={WEATHER_API_KEY}&contentType=json"
        )
        response = requests.get(url)
        response.raise_for_status()
        weather_data = response.json()

        # Save to cache with 12-hour expiry (43200 seconds)
        r.setex(cache_key, 43200, json.dumps(weather_data))

        return jsonify({"source": "api", "data": weather_data})

    except requests.exceptions.RequestException as e:
        return jsonify({"error": str(e)}), 500


In [21]:
from threading import Thread

def run_app():
    app.run(port=5000)

thread = Thread(target=run_app)
thread.start()


 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on http://127.0.0.1:5000
Press CTRL+C to quit
